# Test artery generation & simulation setup

Builds a parametric test artery around an already-skeletonised stent, warps
the stent onto it, checks the mixed-dimensional coupling assumptions, and —
if they hold — writes a runnable 4C input.

This is a **smoke test** of the whole chain, not the physics of the reference
papers: the artery uses a placeholder StVenantKirchhoff material, coupling is
tied meshtying rather than true contact, and the balloon is a simplified
radial point force.

In [1]:
# "sphinx_gallery" renders each Plotly figure as a self-contained text/html
# output (plotly.js loaded from CDN, no dependency on a running kernel), so
# the inline 3D view below also renders interactively on the docs website.
import plotly.io as pio
pio.renderers.default = "sphinx_gallery"

In [2]:
from stentfit import Stent, Artery, Simulation

STENT_NAME    = "stent01"
STENT_DIR     = f'/Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/examples/data/output/stent_skeleton/{STENT_NAME}'
SIM_INPUT_DIR = f'/Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/examples/data/output/simulation'

stent = Stent.load(STENT_DIR, stent_name=STENT_NAME)
artery = Artery(
    stent,
    artery_type="curved",      # "straight" | "curved" | "s_bend"
    inner_margin=0.5,          # extra clearance [mm] between stent and artery inner wall
    wall_thickness=0.5,        # artery wall thickness [mm] (0 = lumen surface only)
    noise_amplitude=0.05,      # fractional wall roughness (0 = smooth pipe)
    noise_seed=0,
    bend_angle_deg=180.0,      # only used by "curved" / "s_bend"
    mesh_type="HEX8",          # "TET4" | "TET10" | "HEX8"
    artery_youngs=2.0,         # wall Young's modulus [MPa]
)
sim = Simulation(
    stent, artery, SIM_INPUT_DIR,

    # Stent beam material and discretisation
    stent_youngs=2.0e5,
    stent_poisson=0.3,
    stent_density=0.0,
    beam_class_label="Beam3rHerm2Line3",

    # Element sizing, both relative to the stent's strut thickness:
    # solid element = strut * factor_solid
    # beam element  = strut * factor_solid * factor_beam
    factor_solid=1.5,
    factor_beam=1.2,

    # Load stepping
    n_steps=10,                # quasi-static steps (force ramps 0 -> full)
    expansion_force=1e-4,      # radial outward force per stent node [N]
)

# One call: align stent → mesh artery solid → assemble → check coupling → write input.
sim.setup()

'''# Or: step by step
sim.align()          # beam mesh + warp onto the centreline
sim.mesh_artery()    # GMSH solid -> artery_solid.4C.yaml
sim.assemble()       # tie beams to the lumen surface
sim.check_coupling() # pass/fail table
sim.write_input()    # only once the checks pass'''

[resume] loading point cloud from /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/examples/data/output/stent_skeleton/stent01/ring_points.csv ...
[resume] restored 10 per-ring 2D skeletons + 1,824,400 surface points. Ready for the manual-edit step.
Artery type      : curved
Artery radius    : 2.072 mm (lumen)
Wall thickness   : 0.500 mm  (outer radius 2.572 mm)
Noise amplitude  : 0.05 (5% of radius)  seed=0
Bend angle       : 180.0 deg
Bend radius      : 7.78 mm  (arc = 24.43 mm)
Arc length       : 27.14 mm  (stent 18.09 mm = 67% of artery)
Centreline       : 150 points  bounds [0. 0. 0.] → [15.55  0.    9.13]
Mesh             : 19,204 vertices  38,400 faces  watertight=True

Stent features
--------------
Loaded stent result from : /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/examples/data/output/stent_skeleton/stent01
Centreline direction     : [1.e-04 1.e+00 0.e+00]
length          :   18.094 mm
diameter        :    3.152 mm
r_outer         :    1.572 mm
strut_

[sim] static smoke test: 10 steps, radial expansion force 0.0001 N ramped over 1,151 stent nodes
[sim] BCs: artery inlet+outlet fixed, one stent node pinned; coupling = beam-to-solid meshtying (tied)
[saved] /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/examples/data/output/simulation/simulation.4C.yaml
[sim] schema-validated. Run in 4C on Linux: set BEAMME_FOUR_C_EXE and launch 4C on this file.

Simulation input ready -> /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/examples/data/output/simulation/simulation.4C.yaml


'# Or: step by step\nsim.align()          # beam mesh + warp onto the centreline\nsim.mesh_artery()    # GMSH solid -> artery_solid.4C.yaml\nsim.assemble()       # tie beams to the lumen surface\nsim.check_coupling() # pass/fail table\nsim.write_input()    # only once the checks pass'

## Results

`setup()` gates the simulation input on the coupling checks — if any fails,
everything up to the assembled mesh is still written, so you can retune the
element sizes and moduli and try again.


In [3]:
print(f"beam elements  : {len(sim.beam_mesh.elements):,}")
print(f"total elements : {len(sim.full_mesh.elements):,}")
print(f"coupling       : {'PASSED' if sim.coupling_report['all_passed'] else 'FAILED'}")
print(f"solid element  : {sim.solid_element_size:.4f} mm")
print(f"beam element   : {sim.beam_element_size:.4f} mm")
print(f"artery solid   : {sim.artery.solid_yaml}")

for path in sorted(sim.sim_input_dir.glob("*.4C.yaml")):
    print(f"  {path.name:26s} {path.stat().st_size / 1e6:8.2f} MB")


beam elements  : 1,016
total elements : 45,632
coupling       : PASSED
solid element  : 0.1609 mm
beam element   : 0.1931 mm
artery solid   : /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/examples/data/output/simulation/artery_solid.4C.yaml
  artery_solid.4C.yaml           8.09 MB
  artery_stent.4C.yaml           8.58 MB
  simulation.4C.yaml             8.84 MB
  stent_warped.4C.yaml           0.42 MB
